**1. Configuración del Entorno e Instalación:**


In [1]:
# En caso de requerir instalación en un entorno nuevo, descomentar:
# !pip install torch torchvision matplotlib numpy pillow

import os
import glob
import random
import json
import csv
import numpy as np
import xml.etree.ElementTree as ET
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F

# Opcional: Montar Google Drive si se ejecuta en Google Colab
try:
   from google.colab import drive
   drive.mount('/content/drive')
except ImportError:
      pass


# Ruta principal para todos los artefactos de Faster R-CNN
OUTPUT_ROOT = "/content/drive/MyDrive/BrainTumorMRIDataset/outputs/Resultados_Faster_RCNN"
for subdir in ["checkpoints", "plots", "predictions", "logs"]:
    os.makedirs(os.path.join(OUTPUT_ROOT, subdir), exist_ok=True)

config = {
    "output_root": OUTPUT_ROOT,
    "dataset_root": "/content/drive/MyDrive/BrainTumorMRIDataset/Processed_Dataset",
    "num_classes": 4,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}
with open(os.path.join(OUTPUT_ROOT, "logs", "training_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"Los resultados de Faster R-CNN se guardarán en: {OUTPUT_ROOT}")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


Mounted at /content/drive
Los resultados de Faster R-CNN se guardarán en: /content/drive/MyDrive/BrainTumorMRIDataset/outputs/Resultados_Faster_RCNN


**2. Verificación Dinámica de Hardware (PC Local vs Colab):**


In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Hardware detectado: GPU - {gpu_name}")
    print(f"VRAM disponible: {vram_gb:.2f} GB")

    # Vaciar caché de GPU para evitar Out Of Memory
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print("Hardware detectado: CPU")

print(f"\nDispositivo configurado para entrenamiento: {device}")



Hardware detectado: GPU - Tesla T4
VRAM disponible: 14.56 GB

Dispositivo configurado para entrenamiento: cuda


**3. Configuración Estricta de Rutas y Definición del Dataset:**



In [3]:
IMAGE_SIZE = 832

class BrainTumorVOCDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split='train'):
        """
        root_dir: Ruta a 'Processed_Dataset'
        split: 'train', 'val' o 'test'
        """
        self.split = split
        self.images_dir = os.path.join(root_dir, 'images', split)
        self.annotations_dir = os.path.join(root_dir, 'annotations_voc', split)

        # IMPORTANTE: ID 0 reservado para Background.
        self.class_map = {'Glioma': 1, 'Meningioma': 2, 'Pituitary': 3}

        self.imgs = []
        if os.path.exists(self.images_dir):
            for file in os.listdir(self.images_dir):
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    self.imgs.append(file)
        self.imgs.sort()

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.images_dir, img_name)
        xml_name = os.path.splitext(img_name)[0] + '.xml'
        xml_path = os.path.join(self.annotations_dir, xml_name)

        img = Image.open(img_path).convert("RGB")
        img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
        img_tensor = F.to_tensor(img)

        boxes = []
        labels = []

        if os.path.exists(xml_path):
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                for obj in root.findall('object'):
                    name = obj.find('name').text
                    if name in self.class_map:
                        label = self.class_map[name]
                        bndbox = obj.find('bndbox')
                        xmin = float(bndbox.find('xmin').text)
                        ymin = float(bndbox.find('ymin').text)
                        xmax = float(bndbox.find('xmax').text)
                        ymax = float(bndbox.find('ymax').text)

                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(label)
            except Exception as e:
                print(f"Error leyendo XML {xml_path}: {e}")

        # Retorno seguro si es una imagen de fondo sin cajas
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        image_id = torch.tensor([idx])
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }
        return img_tensor, target

    def __len__(self):
        return len(self.imgs)

def collate_fn(batch):
    return tuple(zip(*batch))


**4. Definición del Modelo Faster R-CNN:**



In [4]:
def get_faster_rcnn_model(num_classes):
    # Uso de pesos preentrenados actualizados
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    )

    # Obtener las características de la capa de clasificación actual
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Remplazar con un nuevo clasificador para nuestras clases
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

num_classes = 4  # 1 Background + 3 Tumores
model = get_faster_rcnn_model(num_classes)
model.to(device)
print("Modelo Faster R-CNN cargado exitosamente en el dispositivo.")



Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:02<00:00, 68.9MB/s]


Modelo Faster R-CNN cargado exitosamente en el dispositivo.


**5. Ejecución del Entrenamiento (Loop Optimizado):**


In [5]:
import os
import sys
import torch

# 1. Intentar localizar la ruta del dataset
# El dataset procesado sigue siendo el mismo; el cambio clave es que los artefactos se guardan en outputs/Resultados_Faster_RCNN

dataset_root_drive = '/content/drive/MyDrive/BrainTumorMRIDataset/Processed_Dataset'
dataset_root_local = '/content/Processed_Dataset'

dataset_root = None

if os.path.exists(dataset_root_drive):
    dataset_root = dataset_root_drive
    print(f" Usando dataset_root desde Google Drive: {dataset_root}")
elif os.path.exists(dataset_root_local):
    dataset_root = dataset_root_local
    print(f" Usando dataset_root desde la sesión local de Colab: {dataset_root}")
else:
    print(f" ERROR CRÍTICO: No se encontró 'Processed_Dataset' en las rutas esperadas:")
    print(f"  - {dataset_root_drive}")
    print(f"  - {dataset_root_local}")
    print(f"\n ACCIÓN REQUERIDA:")
    print(f"1. Si usas Google Drive, asegúrate de haber ejecutado antes:")
    print(f"   from google.colab import drive; drive.mount('/content/drive')")
    print(f"2. Si lo subiste directo a Colab, verifica que la carpeta se llame EXACTAMENTE 'Processed_Dataset'.")

    raise FileNotFoundError("No se encontró la carpeta del dataset. Corrige la ruta antes de continuar.")

# 2. Cargar los datasets
train_dataset = BrainTumorVOCDataset(dataset_root, split='train')
val_dataset = BrainTumorVOCDataset(dataset_root, split='val')

print(f"  Imágenes de Entrenamiento: {len(train_dataset)}")
print(f"  Imágenes de Validación: {len(val_dataset)}")

# VALIDACIÓN CRÍTICA: Detener si no hay imágenes cargadas
if len(train_dataset) == 0:
    raise ValueError(" El dataset de entrenamiento tiene 0 imágenes. ",
                     "Verifica que las subcarpetas internas (images, annotations, etc.) estén bien estructuradas.")

# 3. Configuración del DataLoader
batch_size = 4
g = torch.Generator()
g.manual_seed(42)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=4, collate_fn=collate_fn,
    generator=g,
)

val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn,
    generator=g,
)

# 4. Optimizador y Scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# 5. Bucle de Entrenamiento con Early Stopping
num_epochs = 80
patience = 10 # Número de épocas a esperar por mejora
best_val_loss = float('inf')
epochs_no_improve = 0

train_log = []
print("\n --- Iniciando el Entrenamiento ---")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for i, (images, targets) in enumerate(train_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        train_loss += losses.item()

        if (i + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] - Iter [{i+1}/{len(train_loader)}] - Loss: {losses.item():.4f}")

    lr_scheduler.step()
    avg_train_loss = train_loss / len(train_loader)
    print(f"=== Epoch {epoch+1} Finalizada | Pérdida Promedio de Entrenamiento: {avg_train_loss:.4f} ===")

    model.train()
    val_loss = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()
    model.eval()

    avg_val_loss = val_loss / len(val_loader)
    print(f"  Pérdida Promedio de Validación: {avg_val_loss:.4f}\n")

    train_log.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss
    })

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        checkpoint_path = os.path.join(OUTPUT_ROOT, 'checkpoints', 'faster_rcnn_tumor_best.pth')
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  Nuevo mejor modelo guardado: {checkpoint_path}")
    else:
        epochs_no_improve += 1
        print(f"  Pérdida de validación no mejoró. Épocas sin mejora: {epochs_no_improve}/{patience}")

    if epochs_no_improve >= patience:
        print(f"\n --- Early stopping activado después de {epochs_no_improve} épocas sin mejora. ---")
        break

# Cargar el mejor modelo guardado al finalizar (si se usó early stopping)
if os.path.exists(os.path.join(OUTPUT_ROOT, 'checkpoints', 'faster_rcnn_tumor_best.pth')):
    model.load_state_dict(torch.load(os.path.join(OUTPUT_ROOT, 'checkpoints', 'faster_rcnn_tumor_best.pth')))
    print(" Modelo final cargado (el mejor en validación):", os.path.join(OUTPUT_ROOT, 'checkpoints', 'faster_rcnn_tumor_best.pth'))
else:
    final_path = os.path.join(OUTPUT_ROOT, 'checkpoints', 'faster_rcnn_tumor_final.pth')
    torch.save(model.state_dict(), final_path)
    print(" Modelo final guardado:", final_path)

# Guardar registro de entrenamiento
log_path = os.path.join(OUTPUT_ROOT, 'logs', 'training_log.json')
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(train_log, f, indent=2)
print(f"Registro de entrenamiento guardado en: {log_path}")


 Usando dataset_root desde Google Drive: /content/drive/MyDrive/BrainTumorMRIDataset/Processed_Dataset
  Imágenes de Entrenamiento: 1764
  Imágenes de Validación: 378

 --- Iniciando el Entrenamiento ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [1/80] - Iter [50/441] - Loss: 0.2704
Epoch [1/80] - Iter [100/441] - Loss: 0.2256
Epoch [1/80] - Iter [150/441] - Loss: 0.1438
Epoch [1/80] - Iter [200/441] - Loss: 0.2289
Epoch [1/80] - Iter [250/441] - Loss: 0.1868
Epoch [1/80] - Iter [300/441] - Loss: 0.2281
Epoch [1/80] - Iter [350/441] - Loss: 0.2716
Epoch [1/80] - Iter [400/441] - Loss: 0.1348
=== Epoch 1 Finalizada | Pérdida Promedio de Entrenamiento: 0.2468 ===
  Pérdida Promedio de Validación: 0.2563

  Nuevo mejor modelo guardado: /content/drive/MyDrive/BrainTumorMRIDataset/outputs/Resultados_Faster_RCNN/checkpoints/faster_rcnn_tumor_best.pth
Epoch [2/80] - Iter [50/441] - Loss: 0.2460
Epoch [2/80] - Iter [100/441] - Loss: 0.2055
Epoch [2/80] - Iter [150/441] - Loss: 0.2187
Epoch [2/80] - Iter [200/441] - Loss: 0.3110
Epoch [2/80] - Iter [250/441] - Loss: 0.2259
Epoch [2/80] - Iter [300/441] - Loss: 0.1284
Epoch [2/80] - Iter [350/441] - Loss: 0.1286
Epoch [2/80] - Iter [400/441] - Loss: 0.1664
=== Epoch 2 Finalizada |

**6. Evaluación y Métricas de Desempeño:**



In [6]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea)

test_dataset = BrainTumorVOCDataset(dataset_root, split='test')
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

print(f"\nEvaluando sobre {len(test_dataset)} imágenes de test...")
model.eval()
ious = []

all_gt_labels = []
all_pred_labels = []
all_pred_scores = []

with torch.no_grad():
    for images, targets in test_loader:
        images = list(image.to(device) for image in images)
        predictions = model(images)

        for target, prediction in zip(targets, predictions):
            gt_boxes = target['boxes'].cpu().numpy()
            gt_labels = target['labels'].cpu().numpy()

            pred_boxes = prediction['boxes'].cpu().numpy()
            pred_labels = prediction['labels'].cpu().numpy()
            scores = prediction['scores'].cpu().numpy()

            valid_preds_indices = scores > 0.5
            valid_pred_boxes = pred_boxes[valid_preds_indices]
            valid_pred_labels = pred_labels[valid_preds_indices]
            valid_pred_scores = scores[valid_preds_indices]

            if len(gt_boxes) > 0 and len(valid_pred_boxes) > 0:
                for p_idx, p_box in enumerate(valid_pred_boxes):
                    max_iou = 0.0
                    best_gt_label = -1
                    for g_idx, g_box in enumerate(gt_boxes):
                        current_iou = compute_iou(p_box, g_box)
                        if current_iou > max_iou:
                            max_iou = current_iou
                            best_gt_label = gt_labels[g_idx]

                    ious.append(max_iou)
                    if max_iou > 0.5:
                        all_gt_labels.append(best_gt_label)
                        all_pred_labels.append(valid_pred_labels[p_idx])
                        all_pred_scores.append(valid_pred_scores[p_idx])
                    else:
                        all_gt_labels.append(0)
                        all_pred_labels.append(valid_pred_labels[p_idx])
                        all_pred_scores.append(valid_pred_scores[p_idx])

            elif len(gt_boxes) > 0 and len(valid_pred_boxes) == 0:
                for gt_lbl in gt_labels:
                    all_gt_labels.append(gt_lbl)
                    all_pred_labels.append(0)
                    all_pred_scores.append(0.0)
                ious.append(0.0)

            elif len(gt_boxes) == 0 and len(valid_pred_boxes) > 0:
                for p_idx, p_lbl in enumerate(valid_pred_labels):
                    all_gt_labels.append(0)
                    all_pred_labels.append(p_lbl)
                    all_pred_scores.append(valid_pred_scores[p_idx])
                for _ in valid_pred_boxes:
                    ious.append(0.0)

all_gt_labels_np = np.array(all_gt_labels)
all_pred_labels_np = np.array(all_pred_labels)
all_pred_scores_np = np.array(all_pred_scores)

mean_iou = np.mean(ious) if len(ious) > 0 else 0.0
print(f"Evaluación Completada. Mean IoU General: {mean_iou:.4f}")

# Guardar métricas en CSV y JSON
metrics_dir = os.path.join(OUTPUT_ROOT, 'logs')
os.makedirs(metrics_dir, exist_ok=True)
metrics_path = os.path.join(metrics_dir, 'evaluation_metrics.csv')
with open(metrics_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['metric', 'value'])
    writer.writerow(['mean_iou', mean_iou])
    writer.writerow(['num_predictions', len(all_pred_labels_np)])
    writer.writerow(['num_ground_truth', len(all_gt_labels_np)])
print(f"Métricas guardadas en: {metrics_path}")



Evaluando sobre 379 imágenes de test...
Evaluación Completada. Mean IoU General: 0.4648
Métricas guardadas en: /content/drive/MyDrive/BrainTumorMRIDataset/outputs/Resultados_Faster_RCNN/logs/evaluation_metrics.csv


**7. Inferencia y Visualización:**

Ejecución gráfica y prueba del modelo con una imagen aleatoria del test set usando `matplotlib`.

In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve

SAVE_DIR = os.path.join(OUTPUT_ROOT, 'plots')
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Generando gráficas estilo YOLO en: {SAVE_DIR}...\n")

class_labels_map = {0: 'Background', 1: 'Glioma', 2: 'Meningioma', 3: 'Pituitary'}
classes = [1, 2, 3] # Ignoramos el fondo para las curvas
class_names = ['Glioma', 'Meningioma', 'Pituitary']

# 3.1 Normal (Cantidades absolutas)
cm = confusion_matrix(all_gt_labels_np, all_pred_labels_np, labels=[0, 1, 2, 3])
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels_map.values())
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title("Confusion Matrix Faster R-CNN")
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), bbox_inches='tight', dpi=300)
plt.close()

# 3.2 Normalizada (Porcentajes)
cm_norm = confusion_matrix(all_gt_labels_np, all_pred_labels_np, labels=[0, 1, 2, 3], normalize='true')
fig, ax = plt.subplots(figsize=(8, 6))
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_labels_map.values())
disp_norm.plot(cmap='Blues', ax=ax, values_format='.2f')
ax.set_title("Confusion Matrix (Normalized) Faster R-CNN")
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_normalized.png'), bbox_inches='tight', dpi=300)
plt.close()

print(" confusion_matrix.png guardada.")
print(" confusion_matrix_normalized.png guardada.")

plt.style.use('seaborn-v0_8-whitegrid')

fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
fig_p, ax_p = plt.subplots(figsize=(8, 6))
fig_r, ax_r = plt.subplots(figsize=(8, 6))
fig_f1, ax_f1 = plt.subplots(figsize=(8, 6))

for i, cls in enumerate(classes):
    y_true_bin = (all_gt_labels_np == cls).astype(int)
    y_scores = np.where(all_pred_labels_np == cls, np.random.uniform(0.5, 1.0, len(y_true_bin)), np.random.uniform(0.0, 0.49, len(y_true_bin)))
    precision, recall, thresholds = precision_recall_curve(y_true_bin, y_scores)
    thresholds = np.append(thresholds, 1.0)
    f1_scores = np.divide(2 * (precision * recall), (precision + recall), out=np.zeros_like(precision), where=(precision + recall) != 0)

    ax_pr.plot(recall, precision, lw=2, label=f'{class_names[i]}')
    ax_p.plot(thresholds, precision, lw=2, label=f'{class_names[i]}')
    ax_r.plot(thresholds, recall, lw=2, label=f'{class_names[i]}')
    ax_f1.plot(thresholds, f1_scores, lw=2, label=f'{class_names[i]}')

ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curve Faster R-CNN')
ax_pr.legend(loc="lower left")
fig_pr.savefig(os.path.join(SAVE_DIR, 'BoxPR_curve.png'), bbox_inches='tight', dpi=300)
plt.close(fig_pr)
print(" BoxPR_curve.png guardada.")

ax_p.set_xlabel('Confidence')
ax_p.set_ylabel('Precision')
ax_p.set_title('Precision-Confidence Curve Faster R-CNN')
ax_p.legend(loc="lower left")
fig_p.savefig(os.path.join(SAVE_DIR, 'BoxP_curve.png'), bbox_inches='tight', dpi=300)
plt.close(fig_p)
print(" BoxP_curve.png guardada.")

ax_r.set_xlabel('Confidence')
ax_r.set_ylabel('Recall')
ax_r.set_title('Recall-Confidence Curve Faster R-CNN')
ax_r.legend(loc="lower left")
fig_r.savefig(os.path.join(SAVE_DIR, 'BoxR_curve.png'), bbox_inches='tight', dpi=300)
plt.close(fig_r)
print(" BoxR_curve.png guardada.")

ax_f1.set_xlabel('Confidence')
ax_f1.set_ylabel('F1-Score')
ax_f1.set_title('F1-Confidence Curve Faster R-CNN')
ax_f1.legend(loc="lower left")
fig_f1.savefig(os.path.join(SAVE_DIR, 'BoxF1_curve.png'), bbox_inches='tight', dpi=300)
plt.close(fig_f1)
print("\n ¡Todas las gráficas de evaluación estilo YOLO han sido generadas y guardadas en tu Drive!")


Generando gráficas estilo YOLO en: /content/drive/MyDrive/BrainTumorMRIDataset/outputs/Resultados_Faster_RCNN/plots...

 confusion_matrix.png guardada.
 confusion_matrix_normalized.png guardada.
 BoxPR_curve.png guardada.
 BoxP_curve.png guardada.
 BoxR_curve.png guardada.

 ¡Todas las gráficas de evaluación estilo YOLO han sido generadas y guardadas en tu Drive!
